Сравню две версии одной модели для распознавания сущностей. Обе обучу на одинаковых предложениях из CoNLL-2003. В одном случае использую метки, сгенерированные LLM, в другом - метки из датасета. Так проверю, как источник разметки влияет на качество

In [1]:
%pip install -q datasets

from datasets import load_dataset

dataset = load_dataset("lhoestq/conll2003")
train = dataset["train"]
validation = dataset["validation"]
test = dataset["test"]

print(dataset)

dataset_infos.json:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.07MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  281kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  259kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


Загрузил CoNLL-2003. Из train возьму предложения для обучения, а validation и test использую для оценки

In [2]:
%pip install -q openai

import json
from pathlib import Path

from google.colab import drive, userdata
from openai import OpenAI

drive.mount("/content/drive")

runs_dir = Path("/content/drive/MyDrive/llm-ner-conll2003/runs")
runs_dir.mkdir(parents=True, exist_ok=True)

client = OpenAI(
    base_url="https://litellm.ispras.ru/v1",
    api_key=userdata.get("LITELLM_API_KEY"),
)

Mounted at /content/drive


Подключил LiteLLM, чтобы получить метки для train, и Google Drive, чтобы сохранить их для обучения маленькой модели

In [3]:
tag_names = [
    "O", "B-PER", "I-PER", "B-ORG", "I-ORG",
    "B-LOC", "I-LOC", "B-MISC", "I-MISC",
]

Взял порядок меток из описания CoNLL-2003. Числовой код метки используется как индекс в этом списке

Перенесу функции из 1conll.ipynb, чтобы размечать train тем же промптом и с той же проверкой ответов, которые использовал для итогового прогона на первом этапе

In [4]:
def make_prompt(tokens):
    return f"""
Label the named entities in this English text using BIO tags.

Entity types:
- PER: person
- ORG: organization
- LOC: location
- MISC: other named entities, including nationality adjectives

Use B-TYPE for the first token of an entity, I-TYPE for its continuation,
and O for tokens outside any entity.

Return only a JSON array of strings, with exactly one tag per input token.
Keep the original token order.

Tokens: {json.dumps(tokens)}
""".strip()

make_prompt - принимает список токенов и собирает из инструкции и токенов одну строку для запроса к GPT. Токены записываю в JSON, чтобы явно передать их границы и порядок. Прошу модель вернуть по одной BIO-метке на токен

In [5]:
def parse_llm_tags(answer, token_count, allowed_tags):
    tags = json.loads(answer)

    if not isinstance(tags, list):
        raise ValueError("Ответ модели должен быть JSON-массивом меток")
    if len(tags) != token_count:
        raise ValueError("Число меток не совпадает с числом токенов")

    previous_tag = "O"

    for tag in tags:
        if tag not in allowed_tags:
            raise ValueError(f"Неизвестная метка: {tag}")

        if tag.startswith("I-"):
            begin_tag = f"B-{tag[2:]}"
            if previous_tag not in (begin_tag, tag):
                raise ValueError(f"Метка {tag} стоит после {previous_tag}")

        previous_tag = tag

    return tags

parse_llm_tags - читает строку ответа как JSON и возвращает список меток, если он прошёл проверку. Проверяю, что меток столько же, сколько токенов, что каждая метка есть в tag_names и что продолжение сущности стоит после начала или продолжения того же типа. Например, I-LOC после O считаю ошибкой. Если модель поставила LOC вместо ORG, ответ может пройти эту проверку, а ошибку в типе покажет сравнение с эталоном

In [6]:
def parse_model_tags(answer, token_count, allowed_tags):
    if not isinstance(answer, str):
        raise ValueError("Модель не вернула строку")

    lines = answer.strip().splitlines()
    if (
        len(lines) >= 2
        and lines[0].strip().lower() in ("```json", "```")
        and lines[-1].strip() == "```"
    ):
        answer = "\n".join(lines[1:-1])

    return parse_llm_tags(answer, token_count, allowed_tags)

parse_model_tags - подготавливает ответ перед проверкой BIO. На первом этапе Gemma иногда присылала массив внутри блока кода Markdown, и json.loads не мог прочитать его целиком. Если в ответе есть открывающая и закрывающая строки такого блока, убираю их и передаю содержимое в parse_llm_tags

In [7]:
def save_answer(run_path, result):
    with run_path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(result, ensure_ascii=False) + "\n")

save_answer - записывает результат одного запроса отдельной строкой в файл JSONL. Открываю файл в режиме добавления, поэтому каждый новый ответ дописывается в конец. Сохраняю номер записи и исходный ответ, чтобы потом найти нужное предложение в train, разобрать ошибки и продолжить разметку после перезапуска Colab

In [8]:
pilot_path = runs_dir / "train_pilot_01.jsonl"
teacher_model = "openai/gpt-oss-120b"
saved_indices = set()

if pilot_path.exists():
    with pilot_path.open(encoding="utf-8") as file:
        for line in file:
            saved_indices.add(json.loads(line)["row_index"])

for row_index in range(5):
    if row_index in saved_indices:
        print(f"Запись {row_index}: ответ уже сохранён")
        continue

    row_tokens = train[row_index]["tokens"]
    response = client.chat.completions.create(
        model=teacher_model,
        messages=[{"role": "user", "content": make_prompt(row_tokens)}],
    )
    answer = response.choices[0].message.content

    predicted_tags = None
    format_error = None

    try:
        predicted_tags = parse_model_tags(answer, len(row_tokens), tag_names)
    except (ValueError, TypeError) as error:
        format_error = str(error)

    save_answer(pilot_path, {
        "row_index": row_index,
        "tokens": row_tokens,
        "model_id": teacher_model,
        "answer": answer,
        "predicted_tags": predicted_tags,
        "format_error": format_error,
    })

    print(f"Запись {row_index}: {format_error or 'метки сохранены'}")

print("Файл:", pilot_path)

Запись 0: ответ уже сохранён
Запись 1: ответ уже сохранён
Запись 2: ответ уже сохранён
Запись 3: ответ уже сохранён
Запись 4: ответ уже сохранён
Файл: /content/drive/MyDrive/llm-ner-conll2003/runs/train_pilot_01.jsonl


Разметил первые пять записей из train, чтобы проверить, как сохраняются ответы GPT перед обучением. Для каждой записи сохранил её номер, исходные токены и ответ модели в поле answer. Если ответ прошёл проверку, записал список меток в predicted_tags. Если проверка не прошла, оставил там None и сохранил причину в format_error

In [9]:
pilot_pairs = []

with pilot_path.open(encoding="utf-8") as file:
    for line in file:
        saved = json.loads(line)
        row_index = saved["row_index"]
        record = train[row_index]

        if saved["tokens"] != record["tokens"]:
            raise ValueError(f"Не совпали токены в записи {row_index}")

        if saved["predicted_tags"] is None:
            print(f"Запись {row_index}: пока нет пригодных меток GPT")
            continue

        gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]

        pilot_pairs.append({
            "row_index": row_index,
            "tokens": record["tokens"],
            "llm_tags": saved["predicted_tags"],
            "gold_tags": gold_tags,
        })

print("Записей для сравнения:", len(pilot_pairs))
print(pilot_pairs[0])

Запись 3: пока нет пригодных меток GPT
Запись 4: пока нет пригодных меток GPT
Записей для сравнения: 3
{'row_index': 0, 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'llm_tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O'], 'gold_tags': ['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']}


По номеру записи нашёл исходное предложение в train и проверил, что токены совпадают с сохранёнными. Для трёх пригодных ответов собрал метки GPT и эталонные метки того же предложения. Две записи с ошибкой длины пока пропустил, чтобы оба набора для обучения содержали одни и те же предложения

In [10]:
import random


def label_train_rows(row_indices, output_path):
    saved_indices = set()

    if output_path.exists():
        with output_path.open(encoding="utf-8") as file:
            for line in file:
                saved_indices.add(json.loads(line)["row_index"])

    for row_index in row_indices:
        if row_index in saved_indices:
            continue

        row_tokens = train[row_index]["tokens"]
        response = client.chat.completions.create(
            model=teacher_model,
            messages=[{"role": "user", "content": make_prompt(row_tokens)}],
        )
        answer = response.choices[0].message.content

        predicted_tags = None
        format_error = None
        try:
            predicted_tags = parse_model_tags(answer, len(row_tokens), tag_names)
        except (ValueError, TypeError) as error:
            format_error = str(error)

        save_answer(output_path, {
            "row_index": row_index,
            "tokens": row_tokens,
            "model_id": teacher_model,
            "answer": answer,
            "predicted_tags": predicted_tags,
            "format_error": format_error,
        })

        saved_indices.add(row_index)


audit_indices = random.Random(42).sample(range(5, len(train)), 50)
audit_path = runs_dir / "train_audit_01.jsonl"

label_train_rows(audit_indices, audit_path)
print("Ответы сохранены в", audit_path)

Ответы сохранены в /content/drive/MyDrive/llm-ner-conll2003/runs/train_audit_01.jsonl


label_train_rows - отправляет записи из train в GPT и сохраняет текст каждого ответа. Если список BIO-меток прошёл проверку длины и правил BIO, функция сохраняет этот список. При ошибке записывает её причину

Выбрал 50 случайных записей с seed 42, исключив первые пять, на которых проверял код. Сохранил ответы в train_audit_01.jsonl. По этому файлу проверю, какая доля ответов пригодна для обучения и насколько метки GPT совпадают с эталоном

Переношу функции из 1conll.ipynb, чтобы сравнить метки GPT с эталоном по тем же правилам, что на первом этапе: сущность засчитывается, если совпали обе границы и тип

In [11]:
def bio_to_spans(tags):
    spans = []
    i = 0

    while i < len(tags):
        tag = tags[i]
        if tag == "O":
            i += 1
            continue

        start = i
        entity_type = tag[2:]
        i += 1

        while i < len(tags) and tags[i] == f"I-{entity_type}":
            i += 1

        spans.append((start, i, entity_type))

    return spans

bio_to_spans - функция, которая собирает целые сущности из BIO-меток, чтобы потом сравнить их границы и типы

In [12]:
def count_matches(gold_tags, predicted_tags):
    if len(gold_tags) != len(predicted_tags):
        raise ValueError("Число меток в двух разметках должно совпадать.")

    gold = set(bio_to_spans(gold_tags))
    predicted = set(bio_to_spans(predicted_tags))

    tp = len(gold & predicted)
    fp = len(predicted - gold)
    fn = len(gold - predicted)

    return tp, fp, fn


def scores_from_counts(tp, fp, fn):
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return precision, recall, f1

В count_matches сравниваю ответ с эталоном и считаю, сколько сущностей нашли правильно (tp - true positive), сколько выделили лишних (fp - false positive) и сколько пропустили (fn - false negative)
В scores_from_counts считаю precision, recall и F1

In [13]:
with audit_path.open(encoding="utf-8") as file:
    audit_results = [json.loads(line) for line in file]

audit_tp, audit_fp, audit_fn = 0, 0, 0
failed_count = 0
failed_gold_count = 0

for result in audit_results:
    record = train[result["row_index"]]
    gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]
    predicted_tags = result["predicted_tags"]

    if predicted_tags is None:
        failed_count += 1
        failed_gold_count += len(bio_to_spans(gold_tags))
        continue

    tp, fp, fn = count_matches(gold_tags, predicted_tags)
    audit_tp += tp
    audit_fp += fp
    audit_fn += fn

_, _, valid_f1 = scores_from_counts(audit_tp, audit_fp, audit_fn)
_, _, full_f1 = scores_from_counts(
    audit_tp, audit_fp, audit_fn + failed_gold_count
)

print(f"Пригодных ответов: {len(audit_results) - failed_count}/{len(audit_results)}")
print(f"Micro-F1 на пригодных ответах: {valid_f1:.2%}")
print(f"Micro-F1 на всех 50 записях: {full_f1:.2%}")
print("Эталонных сущностей в записях без пригодного ответа:", failed_gold_count)

Пригодных ответов: 44/50
Micro-F1 на пригодных ответах: 80.67%
Micro-F1 на всех 50 записях: 68.57%
Эталонных сущностей в записях без пригодного ответа: 21


С первой попытки проверку прошли 44 ответа из 50. На них micro-F1 получился 80,67%. В шести записях без пригодных меток была 21 эталонная сущность. Если учесть их как пропуски, micro-F1 по всей выборке составляет 68,57%

Для шести записей, по которым первый ответ GPT не прошёл проверку, отправлю по одному повторному запросу с тем же промптом. Проверю, для скольких из них удастся получить пригодные метки для обучения

In [14]:
retry_indices = [
    result["row_index"]
    for result in audit_results
    if result["predicted_tags"] is None
]
retry_path = runs_dir / "train_audit_retry_01.jsonl"

label_train_rows(retry_indices, retry_path)

with retry_path.open(encoding="utf-8") as file:
    retry_results = [json.loads(line) for line in file]

recovered_count = sum(
    result["predicted_tags"] is not None
    for result in retry_results
)

print(f"После повтора получили пригодные метки: {recovered_count}/{len(retry_indices)}")

После повтора получили пригодные метки: 5/6


Повторил запрос для шести записей, по которым первый ответ не прошёл проверку. Пять новых ответов прошли проверку, поэтому теперь есть пригодные метки для 49 из 50 записей. Дальше сравню эти пять ответов с эталоном и пересчитаю micro-F1

In [15]:
retry_tp, retry_fp, retry_fn = audit_tp, audit_fp, audit_fn
remaining_failed_gold_count = failed_gold_count

for result in retry_results:
    predicted_tags = result["predicted_tags"]
    if predicted_tags is None:
        continue

    record = train[result["row_index"]]
    gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]

    remaining_failed_gold_count -= len(bio_to_spans(gold_tags))
    tp, fp, fn = count_matches(gold_tags, predicted_tags)
    retry_tp += tp
    retry_fp += fp
    retry_fn += fn

_, _, retry_valid_f1 = scores_from_counts(retry_tp, retry_fp, retry_fn)
_, _, retry_full_f1 = scores_from_counts(
    retry_tp, retry_fp, retry_fn + remaining_failed_gold_count
)

print(f"Micro-F1 на пригодных ответах после повтора: {retry_valid_f1:.2%}")
print(f"Micro-F1 на всех 50 записях после повтора: {retry_full_f1:.2%}")

Micro-F1 на пригодных ответах после повтора: 81.25%
Micro-F1 на всех 50 записях после повтора: 80.25%


Для каждой из шести записей, где первый ответ GPT не прошёл проверку формата и правил BIO, отправил по одному повторному запросу с тем же промптом. При пересчёте метрик использовал первый ответ, если он прошёл проверку, и повторный, если проверку прошёл только он

На записях с пригодными ответами micro-F1 составил 81,25%. На всех 50 записях micro-F1 вырос с 68,57% до 80,25%. Эталонные сущности из записи, для которой обе попытки не прошли проверку, засчитал как пропущенные

При разметке train буду один раз повторять запрос для записей, где первый ответ не прошёл проверку формата и правил BIO

Начинаю собирать разметку для обучения. Перемешиваю индексы train с фиксированным seed и сначала беру 100 записей

Если первый ответ GPT не прошёл проверку формата и правил BIO, отправляю тот же запрос ещё один раз. Ответы двух попыток сохраняю в отдельных файлах

In [18]:
train_order = list(range(len(train)))
random.Random(42).shuffle(train_order)
train_indices = train_order[:100]

train_first_path = runs_dir / "train_stage2_first.jsonl"
train_retry_path = runs_dir / "train_stage2_retry.jsonl"

label_train_rows(train_indices, train_first_path)

with train_first_path.open(encoding="utf-8") as file:
    train_first_results = [json.loads(line) for line in file]

train_retry_indices = [
    result["row_index"]
    for result in train_first_results
    if result["predicted_tags"] is None
]

print(f"Ответов первой попытки в файле: {len(train_first_results)}")
print(f"Записей для повторной попытки: {len(train_retry_indices)}")

label_train_rows(train_retry_indices, train_retry_path)

Ответов первой попытки в файле: 100
Записей для повторной попытки: 8


In [20]:
first_tags_by_index = {
    result["row_index"]: result["predicted_tags"]
    for result in train_first_results
}

retry_tags_by_index = {}
if train_retry_indices:
    with train_retry_path.open(encoding="utf-8") as file:
        for line in file:
            result = json.loads(line)
            retry_tags_by_index[result["row_index"]] = result["predicted_tags"]

train_pairs = []
valid_retry_count = 0

for row_index in train_indices:
    gpt_tags = first_tags_by_index[row_index]

    if gpt_tags is None:
        gpt_tags = retry_tags_by_index[row_index]
        if gpt_tags is None:
            continue
        valid_retry_count += 1

    record = train[row_index]
    train_pairs.append({
        "row_index": row_index,
        "tokens": record["tokens"],
        "gpt_tags": gpt_tags,
        "gold_tags": [tag_names[tag_id] for tag_id in record["ner_tags"]],
    })

print(f"Пригодные метки после повтора: {valid_retry_count}/{len(train_retry_indices)}")
print(f"Записей с двумя версиями разметки: {len(train_pairs)}/{len(train_indices)}")

Пригодные метки после повтора: 5/8
Записей с двумя версиями разметки: 97/100


Подготовлю 97 предложений для DistilBERT и BERT-base. Для каждого бэкбона сопоставлю BIO-метки с подтокенами и проверю, что метки GPT и эталона привязаны к одним и тем же позициям

Проведу пробное обучение DistilBERT и BERT-base с линейным слоем на этих 97 предложениях. Для каждого бэкбона проверю оба источника разметки — GPT и эталон. На этом шаге проверю подготовку данных, запуск обучения и подсчёт micro-F1

Соберу общий обучающий набор из предложений train, для которых получил пригодные метки GPT. Для DistilBERT и BERT-base проверю четыре конфигурации: линейный слой, BiLSTM с линейным слоем и оба этих варианта с CRF. Каждую конфигурацию обучу на одних и тех же предложениях с разметкой GPT и с эталонной разметкой

Сравню варианты на validation по эталонным меткам. Посмотрю, как на micro-F1 влияют выбор бэкбона, добавление BiLSTM и CRF, а также источник разметки

Выберу по validation лучший вариант среди обученных на метках GPT. На test оценю его и модель той же архитектуры, обученную на эталоне. Сравню качество по типам сущностей и разберу конкретные ошибки в границах и типах, а также пропущенные сущности

Начинаю подготовку данных с DistilBERT. На заголовке с LEICESTERSHIRE из validation посмотрю, как токенизатор делит исходные токены на части и как по word_id можно найти исходный токен для каждой части

In [21]:
from transformers import AutoTokenizer

student_checkpoint = "distilbert/distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(student_checkpoint, use_fast=True)

tokens = validation[0]["tokens"]
encoded = tokenizer(tokens, is_split_into_words=True)

for subtoken, word_id in zip(encoded.tokens(), encoded.word_ids()):
    original_token = tokens[word_id] if word_id is not None else None
    print(subtoken, word_id, original_token)

[CLS] None None
CR 0 CRICKET
##IC 0 CRICKET
##KE 0 CRICKET
##T 0 CRICKET
- 1 -
L 2 LEICESTERSHIRE
##EI 2 LEICESTERSHIRE
##CE 2 LEICESTERSHIRE
##ST 2 LEICESTERSHIRE
##ER 2 LEICESTERSHIRE
##S 2 LEICESTERSHIRE
##H 2 LEICESTERSHIRE
##IR 2 LEICESTERSHIRE
##E 2 LEICESTERSHIRE
T 3 TAKE
##A 3 TAKE
##KE 3 TAKE
O 4 OVER
##VE 4 OVER
##R 4 OVER
AT 5 AT
TO 6 TOP
##P 6 TOP
A 7 AFTER
##FT 7 AFTER
##ER 7 AFTER
IN 8 INNINGS
##NI 8 INNINGS
##NG 8 INNINGS
##S 8 INNINGS
VI 9 VICTORY
##CT 9 VICTORY
##OR 9 VICTORY
##Y 9 VICTORY
. 10 .
[SEP] None None


transformers — библиотека Hugging Face для работы с моделями и токенизаторами. Токенизатор разбивает текст на части и переводит их в числа, которые можно передать модели. AutoTokenizer по идентификатору модели подбирает подходящий токенизатор

Чекпоинт — сохранённое состояние модели, в частности её обученные веса. В student_checkpoint храню идентификатор distilbert/distilbert-base-cased. По нему можно найти на Hugging Face и веса модели, и файлы её токенизатора. В этой ячейке AutoTokenizer.from_pretrained загружает словарь и настройки токенизатора. Веса DistilBERT загружу позже, когда буду создавать модель для обучения

use_fast=True просит использовать быструю реализацию токенизатора, если она доступна. Реализация — это код, который выполняет токенизацию. Быстрая версия использует библиотеку на Rust. Она нужна мне ещё и потому, что позволяет через word_ids() узнать, от какого исходного токена получилась каждая часть

is_split_into_words=True говорит токенизатору воспринимать элементы списка как исходные токены одного текста. Без этого параметра список строк будет обработан как несколько отдельных текстов. Например, ["New", "York"] с этим параметром — один текст из двух исходных токенов с индексами 0 и 1. Без параметра это два отдельных текста, и в каждом нумерация начнётся с 0

Теперь использую word_id, чтобы привязать BIO-метку исходного токена к его первой части. Для остальных частей и служебных токенов поставлю -100, чтобы пропустить эти позиции при расчёте ошибки обучения

In [22]:
tag_to_id = {tag: index for index, tag in enumerate(tag_names)}


def align_bio_labels(word_ids, tags):
    labels = []
    previous_word_id = None

    for word_id in word_ids:
        if word_id is None or word_id == previous_word_id:
            labels.append(-100)
        else:
            labels.append(tag_to_id[tags[word_id]])

        previous_word_id = word_id

    return labels


example = train_pairs[0]
encoded = tokenizer(example["tokens"], is_split_into_words=True)
word_ids = encoded.word_ids()

gpt_labels = align_bio_labels(word_ids, example["gpt_tags"])
gold_labels = align_bio_labels(word_ids, example["gold_tags"])

print("Подтокен | Метка GPT | Эталон")
for subtoken, gpt_id, gold_id in zip(encoded.tokens(), gpt_labels, gold_labels):
    gpt_tag = tag_names[gpt_id] if gpt_id != -100 else "—"
    gold_tag = tag_names[gold_id] if gold_id != -100 else "—"
    print(f"{subtoken} | {gpt_tag} | {gold_tag}")

Подтокен | Метка GPT | Эталон
[CLS] | — | —
A | B-PER | B-PER
. | — | —
de | — | —
Silva | I-PER | I-PER
not | O | O
out | O | O
49 | O | O
[SEP] | — | —


align_bio_labels — подготавливает метки для обучения модели. Возвращает список числовых меток для обучения модели. В этом списке по одному значению на каждую позицию после токенизации. Первая часть исходного токена получает числовой код его BIO-метки, а остальные части и служебные токены получают -100, чтобы пропустить их при расчёте ошибки обучения

Функция принимает word_ids и tags. Из word_ids беру индекс исходного токена, по этому индексу нахожу его BIO-метку в tags и через tag_to_id перевожу её в числовой код

Например, если LEICESTERSHIRE стоит на позиции 2 и разбился на три части, в word_ids будет три значения 2 подряд. Для первой части функция берёт tags[2], например B-ORG, и записывает числовой код этой метки. Для следующих двух частей и для служебных токенов записывает -100

Для каждого из 97 предложений беру исходные токены и две готовые разметки — GPT и эталонную. Токенизирую предложение и привязываю каждую BIO-метку к первой части соответствующего токена. Так получаю два обучающих набора с одинаковыми исходными данными и разными метками

In [29]:
from datasets import Dataset

gpt_examples = []
gold_examples = []

for pair in train_pairs:
    encoded = tokenizer(pair["tokens"], is_split_into_words=True)
    word_ids = encoded.word_ids()

    gpt_examples.append({
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": align_bio_labels(word_ids, pair["gpt_tags"]),
    })

    gold_examples.append({
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": align_bio_labels(word_ids, pair["gold_tags"]),
    })

gpt_train_dataset = Dataset.from_list(gpt_examples)
gold_train_dataset = Dataset.from_list(gold_examples)

print(gpt_train_dataset[0])
print("Записей с метками GPT:", len(gpt_train_dataset))
print("Записей с эталонными метками:", len(gold_train_dataset))

{'input_ids': [101, 138, 119, 1260, 11211, 1136, 1149, 3927, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 1, -100, -100, 2, 0, 0, 0, -100]}
Записей с метками GPT: 97
Записей с эталонными метками: 97


input_ids — список числовых кодов из словаря токенизатора, по одному коду на каждую позицию после токенизации. По этим кодам модель берёт векторы из таблицы эмбеддингов

encoded.word_ids() — список индексов исходных токенов CoNLL. Если исходный токен разбился на несколько частей, у всех этих частей будет один и тот же индекс. Для добавленных служебных токенов вроде [CLS] и [SEP] стоит None

encoded.tokens() — список строк после токенизации. В нём видны целые токены, подтокены и добавленные служебные токены, например ["[CLS]", "A", ".", "de", ..., "[SEP]"]

# Как работает DistilBERT

Разберу модель на примере `Hamburg won`. В числовых примерах беру короткие условные векторы, чтобы можно было посчитать всё вручную. У нашего `distilbert-base-cased` на входе и выходе каждого блока Transformer вектор токена содержит **768 координат**. Всего у модели **6 блоков**, и в каждом блоке **12 голов внимания**.

## 1. Таблица эмбеддингов и начальные векторы

Токенизатор переводит каждый токен или подтокен в **числовой код из словаря**. Эти коды хранятся в `input_ids`. Каждый код указывает на строку в **таблице эмбеддингов**. Таблица эмбеддингов — обучаемая матрица, в которой каждая строка содержит вектор одного элемента словаря.

У каждой строки таблицы эмбеддингов DistilBERT 768 координат. Векторы размерности 64 появятся позже, внутри отдельных голов внимания.

У модели также есть таблица **позиционных эмбеддингов**. Из неё берётся вектор позиции токена в предложении. Модель складывает вектор токена и вектор позиции по координатам, применяет LayerNorm и, в режиме обучения, dropout. После этого векторы поступают в первый блок Transformer.

В примере допустим, что после этих операций на вход первого блока пришли такие векторы:

```text
Hamburg → [1, 1]
won     → [3, 2]
```

Для одного и того же токена строка в таблице эмбеддингов одна и та же. Позиционный вектор зависит от места токена в предложении, а после блоков внимания представление зависит и от окружающих токенов.

## 2. Как работает линейный слой

В линейном слое одна строка матрицы содержит веса для одной выходной координаты. При расчёте этой координаты слой:

1. Берёт входные координаты, то есть числа из входного вектора
2. Умножает каждую координату на соответствующий вес из строки матрицы
3. Складывает произведения и прибавляет **смещение** — отдельное обучаемое число для этой выходной координаты

Например, пусть входной вектор равен `[2, 5, 1]`, а строка весов для одной выходной координаты — `[0.5, -0.2, 0.7]`. При смещении `0.1` получаю:

```text
0.5 × 2 + (−0.2) × 5 + 0.7 × 1 + 0.1 = 0.8
```

Числа `[2, 5, 1]` рассчитаны предыдущими операциями модели для текущего токена. Весами и смещением управляет обучение. Для следующей выходной координаты слой берёт тот же входной вектор и другую строку весов со своим смещением.

В математической записи для вектора-столбца это выглядит так:

$$
y = Wx + b
$$

Здесь $x$ — входной вектор, $W$ — матрица весов, $b$ — вектор смещений, $y$ — выходной вектор. **Число столбцов $W$ равно числу входных координат, а число строк равно числу выходных координат.**

## 3. Как из вектора токена получаются Query, Key и Value

**Self-attention, или самовнимание**, позволяет обновить представление каждого токена с учётом других токенов того же предложения. Чтобы выполнить это вычисление, слой применяет к входному вектору токена три отдельных линейных преобразования. У каждого своя матрица весов и своё смещение:

$$
q = W_Qx + b_Q,\qquad
k = W_Kx + b_K,\qquad
v = W_Vx + b_V
$$

- **Query** — вектор токена, для которого сейчас собираю информацию из контекста
- **Key** — вектор, с которым сравниваю Query, чтобы получить оценку для пары токенов
- **Value** — вектор, который затем умножаю на вес внимания и включаю в сумму

Возьму условные преобразования с нулевыми смещениями. Они применяются одинаково к обоим токенам:

| Преобразование | Правило для входа `[a, b]` | `Hamburg` из `[1, 1]` | `won` из `[3, 2]` |
|---|---|---|---|
| Query | `[a, 0]` | `[1, 0]` | `[3, 0]` |
| Key | `[a, b]` | `[1, 1]` | `[3, 2]` |
| Value | `[a, 2 × b]` | `[1, 2]` | `[3, 4]` |

Например, для правила Value матрица равна:

$$
W_V =
\begin{bmatrix}
1 & 0 \\
0 & 2
\end{bmatrix}
$$

Если умножить её на `[3, 2]`, первая строка даст `1 × 3 + 0 × 2 = 3`, а вторая — `0 × 3 + 2 × 2 = 4`. Поэтому Value для `won` получается `[3, 4]`.

## 4. Скалярное произведение, размерность ключа и softmax

Чтобы обновить представление `Hamburg`, сравниваю **Query `Hamburg` с Key каждого токена**, включая сам `Hamburg`. Для сравнения используется **скалярное произведение**: перемножаю координаты на одинаковых позициях и складываю результаты. Из двух векторов получается одно число.

```text
Query Hamburg · Key Hamburg = [1, 0] · [1, 1] = 1 × 1 + 0 × 1 = 1
Query Hamburg · Key won     = [1, 0] · [3, 2] = 1 × 3 + 0 × 2 = 3
```

Оценка для `Hamburg → won` в этом примере выше из-за выбранных чисел. Это оценка вклада `won` при обновлении `Hamburg` в конкретной голове внимания.

Далее делю оценки на $\sqrt{d_k}$, где $d_k$ — **размерность Key**, то есть количество координат. Вектор `[3, 2]` имеет размерность `2`, поэтому здесь делю на $\sqrt{2}$. Масштабирование сдерживает рост величины скалярных произведений при увеличении размерности. Это помогает избежать слишком резкого распределения после softmax.

**Softmax** преобразует набор оценок в веса с суммой `1`. Для каждой оценки $s_i$ вычисляю $e^{s_i}$ и делю на сумму таких значений для всех учитываемых позиций:

$$
\alpha_i = \frac{e^{s_i}}{\sum_j e^{s_j}}
$$

| Шаг | `Hamburg` | `won` |
|---|---:|---:|
| Скалярное произведение | 1 | 3 |
| Деление на $\sqrt{2}$ | ≈ 0.707 | ≈ 2.121 |
| Возведение $e$ в полученную степень | ≈ 2.03 | ≈ 8.34 |
| Деление на общую сумму | ≈ 0.196 | ≈ 0.804 |

Эти числа — **веса внимания для Query `Hamburg`**. Когда модель обновляет `won`, она берёт Query `won` и вычисляет свой набор весов. В реальном входе `attention_mask` также позволяет исключить позиции заполнения `[PAD]` из этого вычисления.

## 5. Что именно суммируется во внимании

Полученные веса умножаю на **векторы Value**, затем складываю результаты по координатам. Для простоты округлю веса до `0.2` и `0.8`.

```text
0.2 × Value Hamburg + 0.8 × Value won
= 0.2 × [1, 2] + 0.8 × [3, 4]
= [0.2, 0.4] + [2.4, 3.2]
= [2.6, 3.6]
```

`[2.6, 3.6]` — результат **одной головы внимания для позиции `Hamburg`** в нашем условном примере. Query и Key использовались для вычисления весов. В сумму вошли взвешенные векторы Value.

**Параметры модели и веса внимания** — разные числа. Матрицы $W_Q$, $W_K$, $W_V$ и их смещения хранятся в модели и могут изменяться при обучении. Веса внимания вроде `0.2` и `0.8` вычисляются заново для конкретного предложения из получившихся Query и Key. Если подать `I live in Hamburg`, модель рассчитает веса уже по токенам этого предложения.

## 6. Головы внимания и объединение их результатов

**Голова внимания** — одно вычисление внимания со своими проекциями Query, Key и Value. Внутри каждого блока DistilBERT таких голов **12**. Количество голов — настройка архитектуры.

В реализации DistilBERT каждый из трёх линейных слоёв для Query, Key и Value сначала преобразует полный входной вектор из 768 чисел в новый вектор из 768 чисел. Затем результат делится на 12 последовательных отрезков по 64 числа. Первая голова получает первые 64 координаты каждой вектора, вторая — следующие 64, и так далее. **Вход каждого преобразования содержит все 768 координат**, поэтому 64 числа одной головы вычислены из полного входного представления.

В одной голове размерность Query и Key равна:

$$
d_k = \frac{768}{12} = 64
$$

Поэтому скалярные произведения в реальной модели делятся на $\sqrt{64}=8$. Каждая голова возвращает для каждого токена вектор из 64 чисел. Эти векторы соединяются **целыми блоками в порядке голов**. На примере трёх голов по две координаты порядок был бы таким:

```text
Голова 1 → [a1, a2]
Голова 2 → [b1, b2]
Голова 3 → [c1, c2]

Объединённый вектор → [a1, a2, b1, b2, c1, c2]
```

У DistilBERT после объединения получается $12 \times 64 = 768$ чисел на токен. Далее **выходной линейный слой внимания** преобразует эти 768 чисел в 768 новых. Каждая выходная координата может использовать входные координаты из разных голов. Так результаты голов объединяются в одно представление токена.

## 7. Остаточное соединение и LayerNorm

После выходного линейного слоя внимания результат складывается по координатам с вектором, который поступил на вход внимания. Такое сложение называется **остаточным соединением**. Оно позволяет передать дальше и прежнее представление токена, и результат внимания.

После сложения применяется **LayerNorm**. Она нормализует координаты **внутри вектора одного токена**. Например, для вектора `[2, 6]` среднее равно `4`, а стандартное отклонение равно `2`. Если опустить маленькую константу $\varepsilon$, которая нужна для устойчивости деления, получаю:

$$
\frac{x_i - \text{среднее}}{\sqrt{\text{дисперсия} + \varepsilon}}
$$
```text
[2, 6] → вычитаем среднее → [−2, 2]
       → делим на стандартное отклонение → [−1, 1]
```

Затем каждая нормализованная координата умножается на свой обучаемый **масштаб** $\gamma$ и получает своё обучаемое **смещение** $\beta$. Например, при масштабе `[2, 3]` и смещении `[0.5, 1]`:

```text
Первая координата:  −1 × 2 + 0.5 = −1.5
Вторая координата:   1 × 3 + 1   =  4
Выход: [−1.5, 4]
```

Среднее и стандартное отклонение вычисляются по текущему вектору. Масштаб и смещение хранятся в модели как обучаемые параметры. LayerNorm нужна, чтобы следующий слой получал векторы с более предсказуемым масштабом. Внутри Transformer результат внимания складывается с входным вектором. Такие сложения и преобразования повторяются в каждом блоке, и величина чисел может меняться от слоя к слою. LayerNorm приводит их к сопоставимому масштабу, что помогает модели обучаться стабильнее. LayerNorm применяется и при обучении, и при оценке.

## 8. FFN — нейросеть внутри блока Transformer

После внимания, остаточного соединения и нормализации вектор поступает в **FFN** (*feed-forward network*, сеть прямого распространения). В нашем блоке FFN состоит из двух линейных слоёв с функцией GELU между ними. **Одна и та же FFN применяется отдельно к каждому токену** в пределах блока. К этому моменту каждый токен уже получил информацию о контексте через внимание.

Первый линейный слой FFN преобразует вектор из **768 координат в 3072**. Его матрица весов содержит **3072 строки по 768 чисел**. Каждая строка даёт одну новую координату. Принцип тот же, что в маленьком примере с двумя входами и тремя выходами:

```text
Вход [2, 3]

Строка весов [1, 0] → 1 × 2 + 0 × 3 = 2
Строка весов [0, 2] → 0 × 2 + 2 × 3 = 6
Строка весов [1, 1] → 1 × 2 + 1 × 3 = 5

Выход [2, 6, 5]
```

После первого линейного слоя к каждой из 3072 координат отдельно применяется **GELU** — нелинейная функция активации. Её можно записать так:

$$
\operatorname{GELU}(x)
= x\Phi(x)
= \frac{x}{2}\left[1+\operatorname{erf}\left(\frac{x}{\sqrt{2}}\right)\right]
$$

Здесь $x$ — одна координата, а $\Phi(x)$ — вероятность того, что величина со стандартным нормальным распределением окажется меньше или равна $x$. **erf** (*error function*, функция ошибок) — математическая функция, которая определяется через интеграл:

$$
\operatorname{erf}(z) = \frac{2}{\sqrt{\pi}} \int_0^z e^{-t^2}\,dt
$$

Через erf выражается функция распределения стандартной нормальной величины:

$$
\Phi(x) = \frac{1}{2}\left[1+\operatorname{erf}\left(\frac{x}{\sqrt{2}}\right)\right]
$$

Например, $\operatorname{erf}(0)=0$, поэтому $\Phi(0)=0.5$ и $\operatorname{GELU}(0)=0 \cdot 0.5=0$ Например, `GELU(−1) ≈ −0.16`, `GELU(0) = 0`, `GELU(1) ≈ 0.84`. В коде это вычисляет готовая функция.

GELU меняет значения координат, сохраняя их количество. Благодаря нелинейности между двумя линейными слоями FFN может выучить преобразование, которое сложнее одного умножения на матрицу с прибавлением смещения. Второй линейный слой возвращает размерность 768:

```text
768 → линейный слой → 3072 → GELU → 3072 → линейный слой → 768
```

После второго линейного слоя FFN применяется dropout. Затем результат складывается с входом FFN через ещё одно остаточное соединение и проходит через LayerNorm.

## 9. Dropout — случайное обнуление при обучении

**Dropout** временно обнуляет случайные компоненты активаций при проходе в режиме обучения. Например, для вектора `[2, 4, 6, 8]` может выпасть маска `[1, 0, 1, 0]`. После умножения по координатам получается `[2, 0, 6, 0]`.

В PyTorch сохранённые значения при обучении дополнительно масштабируются на $1/(1-p)$, где $p$ — вероятность обнуления одной компоненты. При $p=0.5$ это множитель `2`, поэтому выход примера равен `[4, 0, 12, 0]`. При следующем проходе маска может быть другой. Это помогает модели меньше полагаться на отдельные компоненты.

В нашем DistilBERT dropout встроен в модель. В конфигурации чекпоинта `dropout = 0.1` и `attention_dropout = 0.1`. В режиме `model.train()` он срабатывает при проходе автоматически, а в режиме `model.eval()` отключается. Dropout на весах внимания применяется после softmax. Сумма весов равна `1` **до применения dropout**.

## 10. Как всё складывается в шесть блоков и классификацию BIO

**Голова внимания** — одно вычисление с Query, Key и Value. **Блок Transformer** — полный шаг обработки входных векторов. В каждом из шести блоков DistilBERT путь выглядит так:

```text
Входные векторы токенов
→ 12 голов внимания
→ объединение результатов и выходное линейное преобразование
→ остаточное соединение и LayerNorm
→ FFN: линейный слой, GELU, линейный слой и dropout
→ остаточное соединение и LayerNorm
→ обновлённые векторы токенов
```

**Шесть блоков идут последовательно**, а **двенадцать голов работают параллельно внутри каждого блока**. Первый блок получает векторы из эмбеддингов токенов и позиций. Второй получает выход первого и вычисляет из него свои Query, Key и Value. Так продолжается до шестого блока. Каждый блок имеет свои обучаемые параметры и передаёт дальше по одному вектору из 768 чисел на каждую позицию.

Для задачи распознавания сущностей поверх DistilBERT добавляю **линейный слой классификации**. Он получает итоговый вектор позиции из 768 чисел и выдаёт **9 оценок** (9 логитов), по одной на каждую BIO-метку. При предсказании выбираем метку с наибольшим логитом среди всех девяти. При обучении передаём все девять логитов и целевую метку в функцию потерь. Для каждого исходного токена CoNLL целевая метка стоит на его первой части. Остальные части и служебные токены имеют в списке `labels` значение `-100`, поэтому отдельная ошибка для них не считается.

### Источники

- [Конфигурация distilbert-base-cased](https://huggingface.co/distilbert/distilbert-base-cased/blob/main/config.json)
- [Реализация DistilBERT в Hugging Face Transformers](https://github.com/huggingface/transformers/blob/main/src/transformers/models/distilbert/modeling_distilbert.py)
- [Attention Is All You Need, раздел 3](https://arxiv.org/abs/1706.03762)
- [PyTorch — Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html), [LayerNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LayerNorm.html) и [GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html)



Готовлю общий датасет для валидации. Беру все предложения из validation и привязываю эталонные BIO-метки CoNLL-2003 к первым частям исходных токенов. На этом датасете буду считать micro-F1 для обеих моделей

In [30]:
validation_examples = []

for record in validation:
    encoded = tokenizer(record["tokens"], is_split_into_words=True)
    gold_tags = [tag_names[tag_id] for tag_id in record["ner_tags"]]

    validation_examples.append({
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": align_bio_labels(encoded.word_ids(), gold_tags),
    })

validation_dataset = Dataset.from_list(validation_examples)

print("Записей для валидации:", len(validation_dataset))

Записей для валидации: 3250
